# Imports

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

## Helper functions

In [3]:
def describe_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    
    data_types_1 = {}

    for column, datatype in df.dtypes.items():
        datatype_name = datatype.name
        
        if "column" not in data_types_1:
            data_types_1["column"] = []
        
        data_types_1["column"].append(column)
        
        if "data_type" not in data_types_1:
            data_types_1["data_type"] = []
        
        data_types_1["data_type"].append(datatype_name)
        
    df_datatypes_1 = pd.DataFrame(data_types_1)
    
    data = {
        "unique_values_preview": [],
        "num_unique_values": [],
        "num_NaN_values": []
    }

    for column in df_datatypes_1["column"]:
        num_unique_values = df[column].nunique()
        data["num_unique_values"].append(num_unique_values)
        
        if num_unique_values > 3:
            unique_values = ", ".join([str(value) for value in df[column].dropna().unique()[:3]]) + ", ..."
        else:
            unique_values = ", ".join(str(value) for value in df[column].dropna().unique())
            
        data["unique_values_preview"].append(unique_values)
        
        data["num_NaN_values"].append(df[column].isnull().sum())

    df_datatypes_concat = pd.concat([df_datatypes_1, pd.DataFrame(data)], axis=1).sort_values(["data_type", "num_unique_values"]).set_index("data_type")

    return df_datatypes_concat

## Data imports

In [4]:
# CSV imports
cards = pd.read_csv("project_data/cards_data.csv").rename(columns={"id": "card_id"})
users = pd.read_csv("project_data/users_data.csv").rename(columns={"id": "client_id"})
transactions = pd.read_csv("project_data/transactions_data.csv").rename(columns={"id": "transaction_id"})

# JSON imports
series_mcc_codes = pd.read_json("project_data/mcc_codes.json", typ="series")
series_mcc_codes.name = "mcc_code_description"
mcc_codes = pd.DataFrame(series_mcc_codes).reset_index().rename(columns={"index": "mcc_code"})

train_labels = pd.read_json("project_data/train_fraud_labels.json").reset_index().rename(columns={"index": "transaction_id"})

In [5]:
df = transactions \
    .merge(train_labels, on="transaction_id", how="left") \
        .merge(cards.merge(users, on="client_id", how="left"), on="card_id", how="left") \
            .drop("client_id_y", axis=1) \
                .rename(columns={"mcc": "mcc_code", "client_id_x": "client_id"}) \
                    .merge(mcc_codes, on="mcc_code", how="left")

In [6]:
describe_raw_data(df)

,column,unique_values_preview,num_unique_values,num_NaN_values
data_type,,,,
float64,latitude,"46.8, 40.8, 33.18, ...",740,0
float64,longitude,"-100.76, -91.12, -117.29, ...",859,0
float64,zip,"58523.0, 52722.0, 92084.0, ...",25256,1652706
int64,num_cards_issued,"2, 1, 3",3,0
int64,num_credit_cards,"4, 5, 3, ...",9,0
int64,birth_month,"7, 6, 4, ...",12,0
int64,year_pin_last_changed,"2008, 2015, 2006, ...",19,0
int64,retirement_age,"67, 65, 69, ...",27,0
int64,current_age,"30, 48, 49, ...",74,0


# Data Cleaning

In [7]:
mer_state = df[df["merchant_state"].isnull()].index
zip_ = df[df["zip"].isnull()].index

len(set(zip_).intersection(mer_state))

1563700

In [ ]:
df = df.drop(["errors", "card_on_dark_web"], axis=1).drop_duplicates().drop(zip_)

,column,unique_values_preview,num_unique_values,num_NaN_values
data_type,,,,
float64,latitude,"46.8, 40.8, 33.18, ...",740,0
float64,longitude,"-100.76, -91.12, -117.29, ...",859,0
float64,zip,"58523.0, 52722.0, 92084.0, ...",25256,0
int64,num_cards_issued,"2, 1, 3",3,0
int64,num_credit_cards,"4, 5, 3, ...",9,0
int64,birth_month,"7, 6, 4, ...",12,0
int64,year_pin_last_changed,"2008, 2015, 2006, ...",19,0
int64,retirement_age,"67, 65, 69, ...",27,0
int64,current_age,"30, 48, 49, ...",74,0


In [9]:
columns_to_clean = ["per_capita_income", "total_debt", "yearly_income", "credit_limit", "amount"]

for column in columns_to_clean:
    df[column] = df[column].str.replace("$", "").astype("float64")
    
df["date"] = pd.to_datetime(df["date"])

In [11]:
test_set = df[df["target"].isnull()]
df = df.drop(test_set.index)

In [12]:
len(test_set), len(df)

(3845623, 7807586)

In [13]:
describe_raw_data(df)

,column,unique_values_preview,num_unique_values,num_NaN_values
data_type,,,,
datetime64[us],date,"2010-01-01 00:01:00, 2010-01-01 00:02:00, 2010...",3486230,0
float64,latitude,"46.8, 40.8, 33.18, ...",740,0
float64,longitude,"-100.76, -91.12, -117.29, ...",859,0
float64,per_capita_income,"23679.0, 18076.0, 16894.0, ...",1120,0
float64,total_debt,"110153.0, 112139.0, 36540.0, ...",1150,0
float64,yearly_income,"48277.0, 36853.0, 34449.0, ...",1201,0
float64,credit_limit,"55.0, 9100.0, 14802.0, ...",2603,0
float64,zip,"58523.0, 52722.0, 92084.0, ...",24586,0
float64,amount,"-77.0, 14.57, 80.0, ...",64965,0


In [15]:
describe_raw_data(test_set)

,column,unique_values_preview,num_unique_values,num_NaN_values
data_type,,,,
datetime64[us],date,"2010-01-01 00:05:00, 2010-01-01 00:09:00, 2010...",2453029,0
float64,latitude,"41.42, 46.8, 40.77, ...",740,0
float64,longitude,"-87.35, -100.76, -73.84, ...",859,0
float64,per_capita_income,"26168.0, 23679.0, 13810.0, ...",1120,0
float64,total_debt,"128676.0, 110153.0, 375.0, ...",1150,0
float64,yearly_income,"53350.0, 48277.0, 17150.0, ...",1201,0
float64,credit_limit,"37634.0, 55.0, 54.0, ...",2603,0
float64,zip,"46307.0, 58523.0, 11355.0, ...",23015,0
float64,amount,"200.0, 77.0, 10.74, ...",50801,0
